# Heart Disease Prediction: Logistic Regression vs ANN

**NCIT Machine Learning Group Case Study - Complete Google Colab
Notebook**

This notebook implements the complete machine-learning workflow required
by the case-study template:

-   data acquisition and validation;
-   preprocessing without data leakage;
-   exploratory data analysis;
-   a classical model: **Logistic Regression**;
-   a deep-learning model: **Artificial Neural Network (ANN/MLP)**;
-   baseline and tuned versions of both models;
-   structured hyperparameter optimisation;
-   Accuracy, Precision, Recall, F1, Specificity, ROC-AUC and PR-AUC
    comparison;
-   confusion matrices, ROC curves and precision-recall curves;
-   saving trained models; and
-   predicting one new record.

> **Academic-use warning:** This predicts patterns in self-reported
> survey data. It is not a clinical diagnostic system and must not
> replace evaluation by a qualified healthcare professional.

## Why the uploaded 5,000-row file cannot train the model

The uploaded `heart_disease_sample_5000(1).csv` contains the 21
predictors, but it does not contain the answer column
`HeartDiseaseorAttack`. Supervised learning requires known answers
during training. This notebook therefore downloads the correct labeled
BRFSS 2015 version and can create a reproducible, stratified 5,000-row
sample from it.

**Data sources**

-   Cleaned dataset:
    <https://www.kaggle.com/datasets/alexteboul/heart-disease-health-indicators-dataset>
-   Original BRFSS 2015 documentation:
    <https://www.cdc.gov/brfss/annual_data/annual_2015.html>

## How to run

1.  Upload this notebook to Google Colab.
2.  Select **Runtime -\> Run all**.
3.  Keep `USE_SAMPLE = True` for a faster 5,000-row experiment.
4.  Set `USE_SAMPLE = False` if you want to train on all 253,680 cleaned
    records.

The code uses Logistic Regression, not Linear Regression, because the
target is binary (`0` or `1`). Linear Regression is intended for
continuous-number prediction.

## 1. Install and import the required libraries

In [ ]:
# This cell is safe to run more than once.
# Google Colab normally includes TensorFlow. KaggleHub is installed only if missing.
import importlib.util
import subprocess
import sys

required_packages = {
    "kagglehub": "kagglehub",
    "tensorflow": "tensorflow",
}

for module_name, package_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {package_name}...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name]
        )

# Standard libraries
import json
import os
import random
import shutil
import warnings
import zipfile
from pathlib import Path

# Data manipulation and visualisation
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Classical machine learning
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Deep learning
import tensorflow as tf
from tensorflow.keras import Sequential, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, Input

# Dataset downloader
import kagglehub

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

# Reproducibility: the same seed gives repeatable splits and similar training results.
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Setup completed successfully.")

## 2. Configuration and labeled dataset loading

The target means:

-   `0`: respondent did not report coronary heart disease or myocardial
    infarction;
-   `1`: respondent reported coronary heart disease or myocardial
    infarction.

In [ ]:
# ----------------------------- PROJECT SETTINGS -----------------------------
TARGET = "HeartDiseaseorAttack"
KAGGLE_DATASET = "alexteboul/heart-disease-health-indicators-dataset"

# True = use a faster stratified sample. False = use the complete labeled file.
USE_SAMPLE = True
SAMPLE_SIZE = 5000

# Expected inputs, in the exact order used by both trained models.
FEATURES = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "Diabetes", "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump",
    "AnyHealthcare", "NoDocbcCost", "GenHlth", "MentHlth", "PhysHlth",
    "DiffWalk", "Sex", "Age", "Education", "Income"
]


def find_labeled_csv(dataset_directory, target_column):
    """Find the CSV containing the required target column."""
    dataset_directory = Path(dataset_directory)
    csv_files = sorted(dataset_directory.rglob("*.csv"))

    for csv_file in csv_files:
        # Reading only the header is fast and avoids loading unrelated CSV files.
        columns = pd.read_csv(csv_file, nrows=0).columns.tolist()
        if target_column in columns:
            return csv_file

    available = [file.name for file in csv_files]
    raise FileNotFoundError(
        f"No CSV containing '{target_column}' was found. CSV files: {available}"
    )


# KaggleHub downloads public datasets into its local cache.
dataset_directory = kagglehub.dataset_download(KAGGLE_DATASET)
dataset_path = find_labeled_csv(dataset_directory, TARGET)

print("Loading labeled dataset:", dataset_path.name)
df_raw = pd.read_csv(dataset_path)
print("Original shape:", df_raw.shape)
print("Original columns:", df_raw.columns.tolist())

## 3. Validate and clean the data

Important decisions:

-   every required feature and the target are verified;
-   invalid text is converted to missing values;
-   rows with a missing target are removed because their correct class
    is unknown;
-   feature missing values are retained for median imputation after
    splitting;
-   infinite values are treated as missing;
-   duplicate survey responses are reported but retained because
    different respondents can legitimately give identical answers;
-   sampling is stratified so the heart-disease class ratio is
    preserved.

In [ ]:
# Confirm that the dataset is suitable before any modelling begins.
required_columns = FEATURES + [TARGET]
missing_columns = [column for column in required_columns if column not in df_raw.columns]

if missing_columns:
    raise ValueError(
        "The dataset cannot be used for this project. Missing columns: "
        f"{missing_columns}. The uploaded 21-column sample is missing the target "
        f"'{TARGET}', so use the labeled Kaggle file loaded by this notebook."
    )

# Keep only project columns and make an independent copy.
df = df_raw[required_columns].copy()

# Convert every column to numeric. Unexpected text becomes NaN for safe handling.
for column in required_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Replace positive/negative infinity with missing values.
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# A missing target cannot be imputed responsibly, so remove only those rows.
rows_before_target_cleaning = len(df)
df.dropna(subset=[TARGET], inplace=True)
print("Rows removed because target was missing:", rows_before_target_cleaning - len(df))

# This is a binary-classification project: only labels 0 and 1 are valid.
invalid_target_values = sorted(set(df[TARGET].unique()) - {0.0, 1.0})
if invalid_target_values:
    raise ValueError(f"Unexpected target values found: {invalid_target_values}")
df[TARGET] = df[TARGET].astype(int)

# Report duplicates but keep them. Survey datasets can contain legitimate,
# separate respondents with the same set of answers.
exact_duplicate_rows = int(df.duplicated().sum())
print("Exact duplicate rows (retained):", exact_duplicate_rows)

# Take a reproducible stratified sample only after validating the labeled data.
if USE_SAMPLE and len(df) > SAMPLE_SIZE:
    df, _ = train_test_split(
        df,
        train_size=SAMPLE_SIZE,
        stratify=df[TARGET],
        random_state=SEED,
    )

df = df.reset_index(drop=True)

print("\nDataset used for modelling:", df.shape)
print("Total missing feature values:", int(df[FEATURES].isna().sum().sum()))
print("\nTarget counts:")
print(df[TARGET].value_counts().sort_index())
print("\nTarget percentages:")
print((df[TARGET].value_counts(normalize=True).sort_index() * 100).round(2))

## 4. Inspect the data and perform exploratory analysis

These outputs help you explain the dataset during the presentation.
Correlation shows association, not causation.

In [ ]:
# Display sample rows and a compact statistical summary.
display(df.head())
display(df.describe().T)

# Missing-value report before the training-only imputer is fitted.
missing_report = df[FEATURES].isna().sum().sort_values(ascending=False)
missing_report = missing_report[missing_report > 0]
print("Columns with missing values:")
display(missing_report.to_frame("Missing values"))

# Plot target-class balance.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

target_counts = df[TARGET].value_counts().sort_index()
sns.barplot(
    x=["No heart disease/attack", "Heart disease/attack"],
    y=target_counts.values,
    ax=axes[0],
    palette=["#4C78A8", "#E45756"],
)
axes[0].set_title("Target Class Distribution")
axes[0].set_ylabel("Number of respondents")
axes[0].set_xlabel("")

# Show the features having the strongest absolute correlations with the target.
target_correlations = (
    df.corr(numeric_only=True)[TARGET]
    .drop(TARGET)
    .sort_values(key=np.abs, ascending=False)
    .head(12)
    .sort_values()
)
colors = ["#E45756" if value > 0 else "#4C78A8" for value in target_correlations]
axes[1].barh(target_correlations.index, target_correlations.values, color=colors)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Top Feature Correlations with Target")
axes[1].set_xlabel("Pearson correlation")

plt.tight_layout()
plt.show()

## 5. Create training, validation and test sets

The proportions are:

-   60% training: fit baseline models and ANN candidates;
-   20% validation: select ANN hyperparameters;
-   20% test: final unbiased comparison.

Stratification preserves the target ratio. The test set is never used to
fit preprocessing or model parameters.

In [ ]:
X = df[FEATURES].copy()
y = df[TARGET].copy()

# First reserve 20% as the untouched test set.
X_development, X_test, y_development, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)

# Split the remaining 80% into 60% training and 20% validation overall.
X_train, X_validation, y_train, y_validation = train_test_split(
    X_development,
    y_development,
    test_size=0.25,
    stratify=y_development,
    random_state=SEED,
)

print("Training rows:  ", len(X_train))
print("Validation rows:", len(X_validation))
print("Test rows:      ", len(X_test))

for name, labels in [
    ("Training", y_train),
    ("Validation", y_validation),
    ("Test", y_test),
]:
    positive_percentage = labels.mean() * 100
    print(f"{name:10s} positive class: {positive_percentage:.2f}%")

## 6. Define leakage-safe preprocessing and evaluation helpers

Median imputation handles possible missing features. Standardization
changes features to comparable scales, which helps both Logistic
Regression and ANN. Pipelines ensure preprocessing learned from training
data is applied unchanged to validation/test data.

Accuracy alone can be misleading when one class is much larger.
Therefore, we also report Recall, F1, Specificity, ROC-AUC and PR-AUC.

In [ ]:
def make_preprocessor():
    """Return a fresh imputation + scaling pipeline."""
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )


# These containers will build the final comparison table and curves.
metric_records = []
prediction_records = {}


def evaluate_probabilities(model_name, y_true, positive_probabilities, threshold=0.50):
    """Calculate classification metrics from positive-class probabilities."""
    positive_probabilities = np.asarray(positive_probabilities).reshape(-1)
    predictions = (positive_probabilities >= threshold).astype(int)

    # labels=[0, 1] guarantees a consistent 2x2 matrix order.
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "Specificity": specificity,
        "ROC-AUC": roc_auc_score(y_true, positive_probabilities),
        "PR-AUC": average_precision_score(y_true, positive_probabilities),
    }

    metric_records.append(metrics)
    prediction_records[model_name] = {
        "probability": positive_probabilities,
        "prediction": predictions,
    }

    print(f"\n{model_name}")
    print("-" * len(model_name))
    print(pd.Series(metrics).drop("Model").round(4).to_string())
    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            predictions,
            target_names=["No disease/attack", "Disease/attack"],
            zero_division=0,
        )
    )
    return metrics

## 7. Baseline classical model: Logistic Regression

The baseline uses common default settings. It is trained only on the 60%
training set.

In [ ]:
baseline_logistic = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor()),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=SEED,
            ),
        ),
    ]
)

baseline_logistic.fit(X_train, y_train)
baseline_logistic_probabilities = baseline_logistic.predict_proba(X_test)[:, 1]

evaluate_probabilities(
    "Logistic Regression - Baseline",
    y_test,
    baseline_logistic_probabilities,
)

## 8. Tune Logistic Regression with cross-validated grid search

Parameters explored:

-   `C`: inverse regularization strength;
-   `penalty`: L1 or L2 regularization;
-   `solver`: optimization algorithm;
-   `class_weight`: whether the minority class receives greater
    importance.

Five-fold stratified cross-validation is performed only within the 80%
development set. ROC-AUC selects the best setting.

In [ ]:
logistic_for_tuning = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor()),
        (
            "model",
            LogisticRegression(
                max_iter=3000,
                random_state=SEED,
            ),
        ),
    ]
)

# A list of dictionaries prevents invalid solver/penalty combinations.
logistic_parameter_grid = [
    {
        "model__solver": ["liblinear"],
        "model__penalty": ["l1", "l2"],
        "model__C": [0.01, 0.1, 1.0, 10.0],
        "model__class_weight": [None, "balanced"],
    },
    {
        "model__solver": ["lbfgs"],
        "model__penalty": ["l2"],
        "model__C": [0.01, 0.1, 1.0, 10.0],
        "model__class_weight": [None, "balanced"],
    },
]

cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

logistic_search = GridSearchCV(
    estimator=logistic_for_tuning,
    param_grid=logistic_parameter_grid,
    scoring="roc_auc",
    cv=cross_validation,
    n_jobs=-1,
    refit=True,
    verbose=1,
    return_train_score=True,
)

# GridSearchCV learns preprocessing inside each fold, preventing data leakage.
logistic_search.fit(X_development, y_development)
tuned_logistic = logistic_search.best_estimator_

print("Best cross-validation ROC-AUC:", round(logistic_search.best_score_, 4))
print("Best Logistic Regression parameters:")
print(logistic_search.best_params_)

tuned_logistic_probabilities = tuned_logistic.predict_proba(X_test)[:, 1]
evaluate_probabilities(
    "Logistic Regression - Tuned",
    y_test,
    tuned_logistic_probabilities,
)

## 9. Define and train the baseline ANN

ANN structure:

-   input layer: 21 scaled features;
-   hidden Dense layers with ReLU activation;
-   optional dropout and L2 regularization;
-   output layer: one sigmoid neuron producing a probability from 0 to
    1;
-   binary cross-entropy loss for binary classification;
-   Adam optimizer;
-   early stopping to reduce overfitting.

In [ ]:
def build_ann(
    input_dimension,
    hidden_layers=(32, 16),
    dropout_rate=0.0,
    learning_rate=0.001,
    l2_strength=0.0,
):
    """Build and compile a binary-classification artificial neural network."""
    tf.keras.backend.clear_session()

    model = Sequential(name="heart_disease_ann")
    model.add(Input(shape=(input_dimension,), name="patient_features"))

    for layer_number, units in enumerate(hidden_layers, start=1):
        model.add(
            Dense(
                units,
                activation="relu",
                kernel_regularizer=regularizers.l2(l2_strength),
                name=f"hidden_{layer_number}",
            )
        )
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate, name=f"dropout_{layer_number}"))

    # Sigmoid converts the final value into a positive-class probability.
    model.add(Dense(1, activation="sigmoid", name="heart_disease_probability"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
        ],
    )
    return model


# Fit preprocessing only on the training portion.
ann_baseline_preprocessor = make_preprocessor()
X_train_scaled = ann_baseline_preprocessor.fit_transform(X_train).astype("float32")
X_validation_scaled = ann_baseline_preprocessor.transform(X_validation).astype("float32")
X_test_scaled_baseline = ann_baseline_preprocessor.transform(X_test).astype("float32")

baseline_ann = build_ann(
    input_dimension=X_train_scaled.shape[1],
    hidden_layers=(32, 16),
    dropout_rate=0.0,
    learning_rate=0.001,
    l2_strength=0.0,
)

baseline_early_stopping = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=6,
    restore_best_weights=True,
    verbose=1,
)

baseline_ann_history = baseline_ann.fit(
    X_train_scaled,
    y_train.to_numpy(),
    validation_data=(X_validation_scaled, y_validation.to_numpy()),
    epochs=60,
    batch_size=32,
    callbacks=[baseline_early_stopping],
    verbose=1,
)

baseline_ann_probabilities = baseline_ann.predict(
    X_test_scaled_baseline,
    verbose=0,
).reshape(-1)

evaluate_probabilities(
    "ANN - Baseline",
    y_test,
    baseline_ann_probabilities,
)

### Baseline ANN learning curves

Large gaps between training and validation performance may indicate
overfitting.

In [ ]:
history_frame = pd.DataFrame(baseline_ann_history.history)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history_frame["loss"], label="Training loss")
axes[0].plot(history_frame["val_loss"], label="Validation loss")
axes[0].set_title("Baseline ANN Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary cross-entropy")
axes[0].legend()

axes[1].plot(history_frame["auc"], label="Training ROC-AUC")
axes[1].plot(history_frame["val_auc"], label="Validation ROC-AUC")
axes[1].set_title("Baseline ANN ROC-AUC")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("ROC-AUC")
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Tune the ANN with a structured validation search

The search compares network width/depth, dropout, learning rate, L2
regularization, batch size and class weighting. The candidate with the
highest validation ROC-AUC is selected. The test set remains untouched
during selection.

In [ ]:
def balanced_class_weight(labels):
    """Calculate weights inversely proportional to class frequencies."""
    classes = np.array([0, 1])
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=np.asarray(labels),
    )
    return {int(class_label): float(weight) for class_label, weight in zip(classes, weights)}


training_class_weight = balanced_class_weight(y_train)
print("Balanced class weights:", training_class_weight)

# This explicit table makes the tuning process easy to report and reproduce.
ANN_CONFIGURATIONS = [
    {
        "name": "Small balanced",
        "hidden_layers": (32, 16),
        "dropout_rate": 0.10,
        "learning_rate": 0.001,
        "l2_strength": 0.0,
        "batch_size": 32,
        "use_class_weight": True,
    },
    {
        "name": "Medium balanced",
        "hidden_layers": (64, 32),
        "dropout_rate": 0.20,
        "learning_rate": 0.001,
        "l2_strength": 0.0001,
        "batch_size": 32,
        "use_class_weight": True,
    },
    {
        "name": "Deep balanced",
        "hidden_layers": (128, 64, 32),
        "dropout_rate": 0.30,
        "learning_rate": 0.0005,
        "l2_strength": 0.0001,
        "batch_size": 64,
        "use_class_weight": True,
    },
    {
        "name": "Medium unweighted",
        "hidden_layers": (64, 32),
        "dropout_rate": 0.20,
        "learning_rate": 0.001,
        "l2_strength": 0.0001,
        "batch_size": 32,
        "use_class_weight": False,
    },
    {
        "name": "Strong regularization",
        "hidden_layers": (64, 32, 16),
        "dropout_rate": 0.40,
        "learning_rate": 0.0005,
        "l2_strength": 0.001,
        "batch_size": 64,
        "use_class_weight": True,
    },
]

ann_tuning_records = []

for configuration_number, configuration in enumerate(ANN_CONFIGURATIONS, start=1):
    print(f"\nTraining ANN candidate {configuration_number}: {configuration['name']}")

    # Reset seeds so the search is as reproducible as practical.
    random.seed(SEED + configuration_number)
    np.random.seed(SEED + configuration_number)
    tf.random.set_seed(SEED + configuration_number)

    candidate_model = build_ann(
        input_dimension=X_train_scaled.shape[1],
        hidden_layers=configuration["hidden_layers"],
        dropout_rate=configuration["dropout_rate"],
        learning_rate=configuration["learning_rate"],
        l2_strength=configuration["l2_strength"],
    )

    candidate_early_stopping = EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=6,
        restore_best_weights=True,
        verbose=0,
    )

    candidate_history = candidate_model.fit(
        X_train_scaled,
        y_train.to_numpy(),
        validation_data=(X_validation_scaled, y_validation.to_numpy()),
        epochs=60,
        batch_size=configuration["batch_size"],
        class_weight=(
            training_class_weight if configuration["use_class_weight"] else None
        ),
        callbacks=[candidate_early_stopping],
        verbose=0,
    )

    validation_auc_values = candidate_history.history["val_auc"]
    best_epoch = int(np.argmax(validation_auc_values) + 1)
    best_validation_auc = float(np.max(validation_auc_values))

    ann_tuning_records.append(
        {
            **configuration,
            "best_epoch": best_epoch,
            "validation_roc_auc": best_validation_auc,
        }
    )

    print(
        f"Best validation ROC-AUC = {best_validation_auc:.4f} "
        f"at epoch {best_epoch}"
    )

ann_tuning_results = (
    pd.DataFrame(ann_tuning_records)
    .sort_values("validation_roc_auc", ascending=False)
    .reset_index(drop=True)
)
display(ann_tuning_results)

# Select hyperparameters using validation ROC-AUC, never test performance.
best_ann_record = ann_tuning_results.iloc[0].to_dict()
best_ann_name = best_ann_record["name"]
best_ann_configuration = next(
    configuration
    for configuration in ANN_CONFIGURATIONS
    if configuration["name"] == best_ann_name
)
best_ann_epoch = int(best_ann_record["best_epoch"])

print("Selected ANN configuration:", best_ann_name)
print("Retraining epochs:", best_ann_epoch)
print(best_ann_configuration)

### Retrain the selected ANN on all development data

Once hyperparameters and epoch count are selected, the final ANN is
rebuilt using all 80% development rows. Preprocessing is refitted on
that development set, and the 20% test set remains unseen until
evaluation.

In [ ]:
# Refit preprocessing on all development data for the final tuned ANN.
ann_final_preprocessor = make_preprocessor()
X_development_scaled = ann_final_preprocessor.fit_transform(X_development).astype("float32")
X_test_scaled_final = ann_final_preprocessor.transform(X_test).astype("float32")

development_class_weight = balanced_class_weight(y_development)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

tuned_ann = build_ann(
    input_dimension=X_development_scaled.shape[1],
    hidden_layers=best_ann_configuration["hidden_layers"],
    dropout_rate=best_ann_configuration["dropout_rate"],
    learning_rate=best_ann_configuration["learning_rate"],
    l2_strength=best_ann_configuration["l2_strength"],
)

# The epoch count was already chosen on the validation set, so retraining does
# not inspect the test set or require test-based early stopping.
tuned_ann.fit(
    X_development_scaled,
    y_development.to_numpy(),
    epochs=best_ann_epoch,
    batch_size=best_ann_configuration["batch_size"],
    class_weight=(
        development_class_weight
        if best_ann_configuration["use_class_weight"]
        else None
    ),
    verbose=1,
)

tuned_ann_probabilities = tuned_ann.predict(
    X_test_scaled_final,
    verbose=0,
).reshape(-1)

evaluate_probabilities(
    "ANN - Tuned",
    y_test,
    tuned_ann_probabilities,
)

## 11. Master comparison table and model visualisations

The best model depends on the project goal. In imbalanced health
screening data, Recall and PR-AUC are often more informative than
Accuracy alone because they show how well positive cases are found.

In [ ]:
MODEL_ORDER = [
    "Logistic Regression - Baseline",
    "Logistic Regression - Tuned",
    "ANN - Baseline",
    "ANN - Tuned",
]

results_table = pd.DataFrame(metric_records).set_index("Model").loc[MODEL_ORDER].reset_index()

print("MASTER MODEL COMPARISON TABLE")
display(results_table.round(4))

# Plot the main classification metrics.
classification_metrics = ["Accuracy", "Precision", "Recall", "F1", "Specificity"]
plot_data = results_table.melt(
    id_vars="Model",
    value_vars=classification_metrics,
    var_name="Metric",
    value_name="Score",
)

plt.figure(figsize=(14, 6))
sns.barplot(data=plot_data, x="Metric", y="Score", hue="Model")
plt.ylim(0, 1.05)
plt.title("Baseline and Tuned Model Comparison")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Confusion matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for axis, model_name in zip(axes.flat, MODEL_ORDER):
    matrix = confusion_matrix(
        y_test,
        prediction_records[model_name]["prediction"],
        labels=[0, 1],
    )
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=axis,
        xticklabels=["No", "Yes"],
        yticklabels=["No", "Yes"],
    )
    axis.set_title(model_name)
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("Actual class")

plt.suptitle("Test-Set Confusion Matrices", y=1.02, fontsize=15)
plt.tight_layout()
plt.show()

### ROC and precision-recall curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for model_name in MODEL_ORDER:
    probabilities = prediction_records[model_name]["probability"]

    false_positive_rate, true_positive_rate, _ = roc_curve(y_test, probabilities)
    axes[0].plot(
        false_positive_rate,
        true_positive_rate,
        linewidth=2,
        label=f"{model_name} (AUC={roc_auc_score(y_test, probabilities):.3f})",
    )

    precision_values, recall_values, _ = precision_recall_curve(y_test, probabilities)
    axes[1].plot(
        recall_values,
        precision_values,
        linewidth=2,
        label=(
            f"{model_name} "
            f"(AP={average_precision_score(y_test, probabilities):.3f})"
        ),
    )

axes[0].plot([0, 1], [0, 1], "k--", label="Random classifier")
axes[0].set_title("ROC Curves")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate / Recall")
axes[0].legend(fontsize=8)

positive_prevalence = y_test.mean()
axes[1].axhline(
    positive_prevalence,
    color="black",
    linestyle="--",
    label=f"Positive prevalence ({positive_prevalence:.3f})",
)
axes[1].set_title("Precision-Recall Curves")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 12. Interpret Logistic Regression feature coefficients

A positive standardized coefficient increases the model’s log-odds for
the positive class, while a negative coefficient decreases it.
Coefficients describe model associations, not medical causation.

In [ ]:
tuned_logistic_model = tuned_logistic.named_steps["model"]

coefficient_table = pd.DataFrame(
    {
        "Feature": FEATURES,
        "Coefficient": tuned_logistic_model.coef_.reshape(-1),
    }
)
coefficient_table["Absolute coefficient"] = coefficient_table["Coefficient"].abs()
coefficient_table = coefficient_table.sort_values(
    "Absolute coefficient",
    ascending=False,
)

display(coefficient_table.round(4))

top_coefficients = coefficient_table.head(15).sort_values("Coefficient")
coefficient_colors = [
    "#E45756" if value > 0 else "#4C78A8"
    for value in top_coefficients["Coefficient"]
]

plt.figure(figsize=(9, 7))
plt.barh(
    top_coefficients["Feature"],
    top_coefficients["Coefficient"],
    color=coefficient_colors,
)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Top Standardized Logistic Regression Coefficients")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.show()

## 13. Save trained models and experimental results

Saved files:

-   tuned Logistic Regression pipeline, including preprocessing;
-   tuned ANN;
-   ANN preprocessing pipeline;
-   metrics table;
-   Logistic Regression coefficient table;
-   ANN tuning results; and
-   JSON metadata describing features and selected parameters.

In [ ]:
OUTPUT_DIRECTORY = Path("/content/heart_disease_project_outputs")

# Make the path also work outside Colab.
if not Path("/content").exists():
    OUTPUT_DIRECTORY = Path.cwd() / "heart_disease_project_outputs"

if OUTPUT_DIRECTORY.exists():
    shutil.rmtree(OUTPUT_DIRECTORY)
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

# Save the complete classical pipeline (imputer + scaler + model).
joblib.dump(
    tuned_logistic,
    OUTPUT_DIRECTORY / "tuned_logistic_regression.joblib",
)

# Save ANN and its separate preprocessing pipeline.
tuned_ann.save(OUTPUT_DIRECTORY / "tuned_ann.keras")
joblib.dump(
    ann_final_preprocessor,
    OUTPUT_DIRECTORY / "ann_preprocessor.joblib",
)

# Save tables used in the report.
results_table.to_csv(OUTPUT_DIRECTORY / "model_comparison.csv", index=False)
coefficient_table.to_csv(
    OUTPUT_DIRECTORY / "logistic_feature_coefficients.csv",
    index=False,
)
ann_tuning_results.to_csv(
    OUTPUT_DIRECTORY / "ann_tuning_results.csv",
    index=False,
)

# Store everything required to reconstruct the prediction interface.
metadata = {
    "target": TARGET,
    "features_in_order": FEATURES,
    "classification_threshold": 0.50,
    "sample_used": USE_SAMPLE,
    "sample_size": int(len(df)),
    "random_seed": SEED,
    "best_logistic_parameters": logistic_search.best_params_,
    "best_logistic_cv_roc_auc": float(logistic_search.best_score_),
    "best_ann_configuration": best_ann_configuration,
    "best_ann_epoch": best_ann_epoch,
    "academic_use_only": True,
}

with open(OUTPUT_DIRECTORY / "model_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

# Zip the complete output directory for convenient download.
zip_path = shutil.make_archive(
    base_name=str(OUTPUT_DIRECTORY),
    format="zip",
    root_dir=OUTPUT_DIRECTORY,
)

print("Saved output files:")
for output_file in sorted(OUTPUT_DIRECTORY.iterdir()):
    print(" -", output_file.name)
print("ZIP file:", zip_path)

# In Google Colab this opens the browser download dialog.
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab; download the ZIP from the path printed above.")

## 14. Predict a new record

The new record must supply all 21 features using the same coding scheme
as the training data. The function validates the record, applies the
saved preprocessing, and returns probabilities from both tuned models.

The output class uses `0.50` only as an academic default. A real
clinical system would require external validation, calibration,
carefully selected operating thresholds, privacy controls and
professional oversight.

In [ ]:
def predict_new_record(record, logistic_pipeline, ann_model, ann_preprocessor, threshold=0.50):
    """Predict one new survey record with both tuned models."""
    missing_inputs = [feature for feature in FEATURES if feature not in record]
    unexpected_inputs = [key for key in record if key not in FEATURES]

    if missing_inputs:
        raise ValueError(f"Missing input features: {missing_inputs}")
    if unexpected_inputs:
        raise ValueError(f"Unexpected input features: {unexpected_inputs}")

    # Enforce the exact training feature order.
    patient_frame = pd.DataFrame(
        [[record[feature] for feature in FEATURES]],
        columns=FEATURES,
    )
    patient_frame = patient_frame.apply(pd.to_numeric, errors="coerce")

    if patient_frame.isna().any().any():
        bad_columns = patient_frame.columns[patient_frame.isna().any()].tolist()
        raise ValueError(f"Non-numeric or missing values found in: {bad_columns}")

    logistic_probability = float(logistic_pipeline.predict_proba(patient_frame)[0, 1])

    patient_scaled = ann_preprocessor.transform(patient_frame).astype("float32")
    ann_probability = float(ann_model.predict(patient_scaled, verbose=0)[0, 0])

    prediction_table = pd.DataFrame(
        [
            {
                "Model": "Tuned Logistic Regression",
                "Positive-class probability": logistic_probability,
                "Predicted class": int(logistic_probability >= threshold),
            },
            {
                "Model": "Tuned ANN",
                "Positive-class probability": ann_probability,
                "Predicted class": int(ann_probability >= threshold),
            },
        ]
    )
    return prediction_table


# Example values demonstrate the required format only.
new_record = {
    "HighBP": 1,
    "HighChol": 1,
    "CholCheck": 1,
    "BMI": 28,
    "Smoker": 0,
    "Stroke": 0,
    "Diabetes": 0,
    "PhysActivity": 1,
    "Fruits": 1,
    "Veggies": 1,
    "HvyAlcoholConsump": 0,
    "AnyHealthcare": 1,
    "NoDocbcCost": 0,
    "GenHlth": 2,
    "MentHlth": 0,
    "PhysHlth": 2,
    "DiffWalk": 0,
    "Sex": 1,
    "Age": 8,
    "Education": 5,
    "Income": 6,
}

new_record_prediction = predict_new_record(
    record=new_record,
    logistic_pipeline=tuned_logistic,
    ann_model=tuned_ann,
    ann_preprocessor=ann_final_preprocessor,
    threshold=0.50,
)

display(new_record_prediction)

## 15. How to explain the complete workflow

1.  **Problem:** Predict whether a respondent reported coronary heart
    disease or a heart attack.
2.  **Target:** `HeartDiseaseorAttack`, where `0` means no and `1` means
    yes.
3.  **Preprocessing:** Validate schema, convert to numeric, reserve the
    test set, median-impute missing features and standardize inputs.
4.  **Classical model:** Logistic Regression provides a simple,
    interpretable probability model.
5.  **Deep model:** ANN learns nonlinear relationships through hidden
    Dense layers.
6.  **Fine-tuning:** Grid search tunes Logistic Regression with
    five-fold cross-validation; a structured validation search tunes ANN
    architecture and training parameters.
7.  **Evaluation:** Compare baseline and tuned models on the same
    untouched test set using several metrics and curves.
8.  **Deployment preparation:** Save both tuned models, preprocessing
    and metadata; validate all 21 fields before predicting a new record.
9.  **Limitation:** The data are self-reported and observational. A good
    test score does not make the model a clinical diagnosis tool.